In [ ]:
import pandas as pd
import geopandas as gpd

from rasterstats import zonal_stats
import rasterio as rio

In [ ]:
shp=gpd.read_file(snakemake.input.euroshape).to_crs("EPSG:3035")
corine=snakemake.input.corine

In [ ]:
stats = pd.DataFrame(zonal_stats(vectors=shp['geometry'], raster=corine, categorical=True,stats='count'))

In [ ]:
ras = rio.open(corine)
a = ras.transform
x, y = abs(a[0]), abs(a[4])
pixelarea = x*y

stats = stats.fillna(0)*pixelarea/1e6 #Area in sq km per pixel value

In [ ]:
result = (pd.merge(left=shp.drop("geometry",axis=1), 
                      right=stats, 
                      left_index=True, 
                      right_index=True)
         )

if "CNTR_agg" in result.columns:
    result=result.set_index(["zone","index","CNTR","CNTR_agg"])
else:
    result=result.set_index(["zone","index","CNTR"])

result=result.loc[:,[1,2,3]].sum(axis=1)

area=(pd.read_csv(snakemake.input.country_rooftop_areas)
            .rename(columns={"zone":"CNTR"})
            .set_index("CNTR"))

out=(result.div(
        result.groupby("CNTR").sum())
        .mul(area["useable roof surface (km2)"])
        .reset_index())

if "CNTR_agg" in out.columns:
    out=(out.groupby("CNTR_agg").sum()
        .assign(spatial=lambda x: x.index)
        .reset_index()
        .loc[:,["CNTR_agg","spatial",0]]
        .rename(columns={"CNTR_agg":"zone",0:"area"}))
else:
    out=(out.rename(columns={"index":"spatial",0:"area"})
        .loc[:,["zone","spatial","area"]])

out.to_csv(snakemake.output.rooftop_areas,index=False)